# L2Flow Instrument History / Latest / Event Polars 示例

这个 notebook 在 C++ 后端运行后，通过 Python 客户端读取一个 instrument 的：

- 当前交易日 catalog 的完整 `symbols` 字典；
- 以 symbol 为输入读取 Instrument Store 固定 generation 中的历史 snapshot 与 tick；
- latest snapshot / latest tick，以及历史中最近一笔真正的成交；
- 由 tick/order/transaction 推导出的 instrument history event。

示例同时说明 `partial`、普通 `from-open` 和 `from-open recovery` 三种 coverage。代码只读取已有数据，不会修改后端。

## 接口概览

| 调用 | 主要返回值 | 用途 |
|---|---|---|
| `find_and_connect_l2flow_backend()` | `L2FlowClient` + `Path` + `SessionInfo` | 查找当前 C++ 后端并完成真实 Wire V2 握手 |
| `connect(socket, native_library=...)` | `L2FlowClient` | 连接指定的 C++ Wire V2 控制面 |
| `client.session_info()` | `SessionInfo` | session 状态、交易日、数据计数、recovery 标志 |
| `client.history_coverage()` | `HistoryCoverageInfo` | `FROM_OPEN` / `PROCESS_START_PARTIAL` / `UNAVAILABLE` |
| `build_symbol_dictionary(client)` | `symbols` 字典 + `symbols_frame` | 取得 `CATALOG_ALL` 的 symbol 全集 |
| `latest_snapshot_by_symbol(frames, symbol, symbols)` | 单行 `polars.DataFrame` | 按 symbol 读取最近 snapshot |
| `latest_tick_by_symbol(frames, symbol, symbols)` | 单行 `polars.DataFrame` | 按 symbol 读取最近 tick-like 记录 |
| `read_retained_fast_ticks_by_symbol(...)` | scan metadata + tick DataFrame | partial recovery 期间读取 bounded FAST ring |
| `read_store_history_by_symbol(...)` | generation + snapshot/tick DataFrames | 按 symbol 读取固定 Store generation |
| `open_derived_event_history_by_symbol(...)` | history handle；`latest_snapshot().dataframe` | 按 symbol 读取 derived events |

`latest_tick()` 返回最近的 tick-like 记录，可能是下单、撤单、状态或成交，并不保证是成交。若需要“最近一笔真正成交”，应从完整 tick history 中筛选 `action == TickAction.TRADE`。

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import polars as pl
from IPython.display import display

from l2flow_realtime import (
    InconsistentReadError,
    SelectionScope,
    TickAction,
    TickOverrunError,
    UnavailableError,
    as_polars,
    connect,
)
from l2flow_realtime.polars import PolarsHistoryCoverageError


def retry_inconsistent_read(operation, maximum_attempts: int = 20):
    last_error = None
    for _ in range(maximum_attempts):
        try:
            return operation()
        except InconsistentReadError as error:
            last_error = error
    assert last_error is not None
    raise last_error


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python/l2flow_realtime").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())

native_from_env = os.environ.get("L2FLOW_SHM_READER_LIBRARY")
native_candidates = (
    Path(native_from_env).expanduser() if native_from_env else None,
    REPO_ROOT / "build/libl2flow_shm_reader.so",
    REPO_ROOT / "build-live-latest/libl2flow_shm_reader.so",
    REPO_ROOT / "build-live-current-20260803/libl2flow_shm_reader.so",
)
NATIVE_LIBRARY = next(
    (path.resolve() for path in native_candidates if path and path.is_file()),
    None,
)
CONNECT_OPTIONS = (
    {"native_library": str(NATIVE_LIBRARY)} if NATIVE_LIBRARY else {}
)

BACKEND_EXECUTABLE_NAMES = {
    "mdl-production-router",
    "mdl_production_router",
}
CONTROL_SOCKET_OPTIONS = (
    "--ipc-socket",
    "--live-preview-ipc-socket",
)
DEFAULT_SOCKET_SEARCH_ROOTS = (
    REPO_ROOT / "artifacts",
    Path("/run/l2flow"),
    Path(f"/run/user/{os.getuid()}/l2flow"),
    Path("/tmp/l2flow"),
)
TEMP_SOCKET_PATTERNS = (
    "l2flow*.sock",
    "*/l2flow*.sock",
    "*/fast.sock",
    "*/preview.sock",
    "*/live-preview.sock",
    "*/recovered.sock",
)


class BackendDiscoveryError(ConnectionError):
    pass


def normalize_socket_path(value: str | os.PathLike[str]) -> Path:
    return Path(value).expanduser().resolve()


def running_backend_socket_options() -> list[dict[str, Path]]:
    """Read control socket arguments from running router processes."""
    processes: list[dict[str, Path]] = []
    for cmdline_path in Path("/proc").glob("[0-9]*/cmdline"):
        try:
            raw_arguments = cmdline_path.read_bytes().split(b"\0")
        except OSError:
            continue
        arguments = [
            value.decode(errors="replace")
            for value in raw_arguments
            if value
        ]
        if not arguments:
            continue
        if not any(
            Path(argument).name in BACKEND_EXECUTABLE_NAMES
            for argument in arguments
        ):
            continue

        endpoints: dict[str, Path] = {}
        for index, argument in enumerate(arguments):
            for option in CONTROL_SOCKET_OPTIONS:
                if argument == option and index + 1 < len(arguments):
                    endpoints[option] = normalize_socket_path(
                        arguments[index + 1]
                    )
                elif argument.startswith(f"{option}="):
                    endpoints[option] = normalize_socket_path(
                        argument.split("=", 1)[1]
                    )
        if endpoints:
            processes.append(endpoints)
    return processes


def is_fast_control_socket_candidate(path: Path) -> bool:
    name = path.name.lower()
    return name.endswith(".sock") and not any(
        marker in name
        for marker in ("event", "delta", "certified")
    )


def endpoint_name_priority(path: Path) -> int:
    name = path.name.lower()
    if "recovered" in name:
        return 30
    if "preview" in name:
        return 20
    return 10


def socket_mtime_ns(path: Path) -> int:
    try:
        return path.stat().st_mtime_ns
    except OSError:
        return 0


def discover_l2flow_control_sockets(
    *,
    explicit_socket: str | os.PathLike[str] | None = None,
    search_roots=None,
) -> list[Path]:
    """Return likely FAST control sockets in connection order."""
    ranks: dict[Path, tuple[int, int]] = {}

    def add_candidate(
        value,
        source_priority: int,
        name_priority: int,
        *,
        require_socket: bool,
        require_control_name: bool,
    ) -> None:
        if value is None or str(value).strip() == "":
            return
        path = normalize_socket_path(value)
        if (
            require_control_name
            and not is_fast_control_socket_candidate(path)
        ):
            return
        try:
            if require_socket and not path.is_socket():
                return
        except OSError:
            return
        rank = (source_priority, name_priority)
        ranks[path] = max(ranks.get(path, (0, 0)), rank)

    configured_socket = (
        explicit_socket
        if explicit_socket is not None
        else os.environ.get("L2FLOW_CONTROL_SOCKET")
    )
    add_candidate(
        configured_socket,
        3,
        0,
        require_socket=False,
        require_control_name=False,
    )

    for endpoints in running_backend_socket_options():
        add_candidate(
            endpoints.get("--ipc-socket"),
            2,
            30,
            require_socket=False,
            require_control_name=False,
        )
        add_candidate(
            endpoints.get("--live-preview-ipc-socket"),
            2,
            20,
            require_socket=False,
            require_control_name=False,
        )

    roots = (
        DEFAULT_SOCKET_SEARCH_ROOTS
        if search_roots is None
        else tuple(Path(root) for root in search_roots)
    )
    for root in roots:
        root = normalize_socket_path(root)
        try:
            if root.is_socket():
                add_candidate(
                    root,
                    1,
                    endpoint_name_priority(root),
                    require_socket=True,
                    require_control_name=True,
                )
            elif root.is_dir():
                for path in root.rglob("*.sock"):
                    add_candidate(
                        path,
                        1,
                        endpoint_name_priority(path),
                        require_socket=True,
                        require_control_name=True,
                    )
        except OSError:
            continue

    temporary_root = Path("/tmp")
    for pattern in TEMP_SOCKET_PATTERNS:
        try:
            temporary_candidates = temporary_root.glob(pattern)
            for path in temporary_candidates:
                add_candidate(
                    path,
                    1,
                    endpoint_name_priority(path),
                    require_socket=True,
                    require_control_name=True,
                )
        except OSError:
            continue

    return sorted(
        ranks,
        key=lambda path: (
            ranks[path][0],
            socket_mtime_ns(path),
            ranks[path][1],
        ),
        reverse=True,
    )


def find_and_connect_l2flow_backend(
    *,
    explicit_socket: str | os.PathLike[str] | None = None,
    search_roots=None,
    timeout: float = 0.5,
    native_library: str | os.PathLike[str] | None = NATIVE_LIBRARY,
):
    """Discover, handshake, validate, and return client/path/session."""
    candidates = discover_l2flow_control_sockets(
        explicit_socket=explicit_socket,
        search_roots=search_roots,
    )
    failures: list[str] = []

    for socket_path in candidates:
        candidate_client = None
        try:
            options = {"timeout": timeout}
            if native_library is not None:
                options["native_library"] = str(native_library)
            candidate_client = connect(str(socket_path), **options)
            candidate_session = retry_inconsistent_read(
                candidate_client.session_info
            )
            return candidate_client, socket_path, candidate_session
        except Exception as error:
            if candidate_client is not None:
                candidate_client.close()
            message = " ".join(str(error).splitlines())
            failures.append(
                f"{socket_path}: {type(error).__name__}: {message[:240]}"
            )

    if not candidates:
        raise BackendDiscoveryError(
            "没有找到 L2Flow control socket。请先启动 "
            "mdl-production-router，或设置 L2FLOW_CONTROL_SOCKET。"
        )
    raise BackendDiscoveryError(
        "发现了候选 socket，但都未通过 L2Flow Wire V2 握手：\n- "
        + "\n- ".join(failures)
    )


def recovered_control_socket_for(current_socket: Path) -> Path | None:
    configured = os.environ.get("L2FLOW_RECOVERED_CONTROL_SOCKET")
    if configured:
        return normalize_socket_path(configured)

    current_socket = normalize_socket_path(current_socket)
    for endpoints in running_backend_socket_options():
        preview = endpoints.get("--live-preview-ipc-socket")
        recovered = endpoints.get("--ipc-socket")
        if (
            preview is not None
            and recovered is not None
            and current_socket in (preview, recovered)
        ):
            return recovered

    if current_socket.name in ("preview.sock", "live-preview.sock"):
        return current_socket.with_name("recovered.sock")
    if "recovered" in current_socket.name.lower():
        return current_socket
    return None

# 所有数据读取都使用这个 symbol。格式为 SH.600000 / SZ.000001。
SYMBOL = os.environ.get("L2FLOW_SYMBOL", "SH.600000").strip().upper()

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(12)

## 连接与 coverage 判断

`find_and_connect_l2flow_backend()` 会先读取运行中 `mdl-production-router` 的 `--ipc-socket` / `--live-preview-ipc-socket` 参数，再扫描仓库 `artifacts/` 和常见 runtime 目录。候选路径必须实际完成 Wire V2 握手并通过 `session_info()` 验证才会被采用；残留的旧 socket 文件会被跳过。函数返回 `(client, CONTROL_SOCKET, session)`。如需固定端点，可设置 `L2FLOW_CONTROL_SOCKET`，它会被优先尝试。

这里不要只看 `server_state`，还要同时看 `history_coverage()`：

- 普通 from-open：`coverage_from_open=True` 且 `startup_prefix_recovered=False`；
- standalone partial：`coverage_kind=PROCESS_START_PARTIAL`，首个 Store generation 发布后 History 可读；
- online recovery 的 partial preview：latest 可读，但 coverage 为 `UNAVAILABLE`，History/event history 会被服务端拒绝；
- from-open recovery：promotion 后切换到 recovered socket，`coverage_from_open=True` 且 `startup_prefix_recovered=True`。

In [ ]:
client, CONTROL_SOCKET, session = find_and_connect_l2flow_backend()
RECOVERED_CONTROL_SOCKET = recovered_control_socket_for(CONTROL_SOCKET)
frames = as_polars(client)
coverage = retry_inconsistent_read(client.history_coverage)

if coverage.coverage_from_open:
    coverage_mode = (
        "from_open_recovery"
        if session.startup_prefix_recovered
        else "from_open"
    )
elif coverage.process_start_partial:
    coverage_mode = "partial"
else:
    coverage_mode = "partial_recovery_preview"

session_frame = pl.DataFrame(
    {
        "control_socket": [str(CONTROL_SOCKET)],
        "server_state": [session.server_state.name],
        "coverage_mode": [coverage_mode],
        "trade_date": [session.trade_date],
        "session_epoch": [session.session_epoch],
        "run_id": [session.run_id.hex()],
        "available_count": [session.available_count],
        "snapshot_available_count": [
            session.snapshot_available_count
        ],
        "tick_available_count": [session.tick_available_count],
        "coverage_from_open": [session.coverage_from_open],
        "startup_prefix_recovered": [
            session.startup_prefix_recovered
        ],
    }
)
# symbols 字典会在下一个单元首先输出，随后再展示这些状态字段。

## 完整 symbols 字典（当前日 catalog 全集）

`SelectionScope.CATALOG_ALL` 返回完整日 catalog 的 session-scoped IDs。下面逐项读取 catalog 元数据，构造完整 Python 字典 `symbols`：键使用不会跨市场冲突的 `SH.600000` / `SZ.000001`，值包含内部 instrument ID、市场、原始证券代码、source 和状态。`symbols_frame` 是同一全集的 Polars 表。

为避免 notebook 一次渲染 5200 多行，Polars 只显示截断预览；`symbols` 和 `symbols_frame` 变量本身都包含全集。后续所有读取单元只接受 `SYMBOL`，instrument ID 仅在 helper 内部解析。

In [ ]:
MARKET_PREFIX = {1: "SH", 2: "SZ"}


def decode_catalog_bytes(value: bytes | None) -> str:
    return "" if value is None else value.decode(errors="replace")


def build_symbol_dictionary(client):
    catalog = retry_inconsistent_read(
        lambda: client.select(SelectionScope.CATALOG_ALL)
    )
    symbols: dict[str, dict[str, object]] = {}
    rows: list[dict[str, object]] = []

    for instrument_id in catalog.instrument_ids:
        instrument = retry_inconsistent_read(
            lambda instrument_id=instrument_id: client.instrument(
                instrument_id
            )
        )
        market_prefix = MARKET_PREFIX.get(instrument.market)
        if market_prefix is None:
            raise RuntimeError(
                f"instrument {instrument_id} 使用未知市场 {instrument.market}"
            )
        security_id = decode_catalog_bytes(instrument.security_id)
        symbol = f"{market_prefix}.{security_id}"
        if symbol in symbols:
            raise RuntimeError(f"catalog 中出现重复 symbol: {symbol}")

        metadata = {
            "instrument_id": instrument.instrument_id,
            "market": market_prefix,
            "security_id": security_id,
            "security_id_source": decode_catalog_bytes(
                instrument.security_id_source
            ),
            "status": instrument.status.name,
        }
        symbols[symbol] = metadata
        rows.append({"symbol": symbol, **metadata})

    if len(symbols) != catalog.returned_row_count:
        raise RuntimeError("symbols 数量与 CATALOG_ALL 不一致")
    return symbols, pl.DataFrame(rows).sort(["market", "security_id"])


def instrument_id_for(
    symbol: str, symbol_dictionary: dict[str, dict[str, object]]
) -> int:
    normalized = symbol.strip().upper()
    try:
        return int(symbol_dictionary[normalized]["instrument_id"])
    except KeyError as error:
        raise KeyError(
            f"未知 symbol {normalized!r}；请先在 symbols_frame 中查询"
        ) from error


symbols, symbols_frame = build_symbol_dictionary(client)
print(f"symbols 全集数量: {len(symbols)}")
display(symbols_frame)

instrument_id_for(SYMBOL, symbols)  # 只在内部转换；后续输入仍为 SYMBOL
display(pl.DataFrame([{"symbol": SYMBOL, **symbols[SYMBOL]}]))

print("current endpoint state")
display(session_frame)
display(frames.history_coverage())

## 最近 snapshot 与 tick

这两个 symbol helper 内部调用 latest 热读路径，online recovery 的 partial preview 也可以使用。返回值已经是 Polars DataFrame；`status == 0` 表示 `LatestStatus.AVAILABLE`。价格使用规范化的 `*_p6` 整数列，真实价格为该值除以 `1_000_000`。

In [ ]:
def latest_snapshot_by_symbol(frames, symbol, symbol_dictionary):
    normalized = symbol.strip().upper()
    frame = frames.latest_snapshot(
        instrument_id_for(normalized, symbol_dictionary)
    )
    return frame.with_columns(
        pl.lit(normalized).alias("symbol")
    ).select("symbol", *frame.columns)


def latest_tick_by_symbol(frames, symbol, symbol_dictionary):
    normalized = symbol.strip().upper()
    frame = frames.latest_tick(
        instrument_id_for(normalized, symbol_dictionary)
    )
    return frame.with_columns(
        pl.lit(normalized).alias("symbol")
    ).select("symbol", *frame.columns)


latest_snapshot = latest_snapshot_by_symbol(
    frames, SYMBOL, symbols
).select(
    "symbol",
    "session_epoch",
    "instrument_id",
    "status",
    "ingress_sequence",
    "event_time_unix_ns",
    "last_price_p6",
)
latest_tick = latest_tick_by_symbol(frames, SYMBOL, symbols).select(
    "symbol",
    "session_epoch",
    "instrument_id",
    "status",
    "ingress_sequence",
    "tick_stream_sequence",
    "event_time_unix_ns",
    "event_kind",
    "action",
    "side",
    "price_p6",
    "quantity_raw",
)

print("latest snapshot")
display(latest_snapshot)
print("latest tick-like record; TickAction.TRADE =", int(TickAction.TRADE))
display(latest_tick)

## Partial recovery：读取 bounded FAST tick ring

Online recovery preview 虽然尚未开放不可变 Store History，但已经开放 FAST latest 和全市场 bounded tick ring。`client.open_fast_tick_stream(expected_sequence=...)` 可以读取 ring 当前仍保留的 tick/order/transaction 数据。

下面默认扫描 frontier 之前最近 65536 条 retained FAST 记录，按 `SYMBOL` 过滤并转换为 Polars。生产者仍在写入时可能发生 `TickOverrunError` 或瞬时 `InconsistentReadError`；示例会在有界次数内重试。这个 lookback 结果只是当前 ring retention，不是 from-open 完整历史。

In [ ]:
FAST_TICK_SCHEMA = {
    "instrument_id": pl.UInt32,
    "ingress_sequence": pl.UInt64,
    "source_sequence": pl.UInt64,
    "tick_stream_sequence": pl.UInt64,
    "event_time_unix_ns": pl.Int64,
    "recv_realtime_ns": pl.Int64,
    "event_kind": pl.UInt8,
    "market": pl.UInt8,
    "action": pl.UInt8,
    "side": pl.UInt8,
    "price_p6": pl.Int64,
    "price_valid": pl.UInt8,
    "price_is_null": pl.UInt8,
    "quantity_raw": pl.Int64,
    "quantity_valid": pl.UInt8,
    "quantity_is_null": pl.UInt8,
    "trade_amount_p6": pl.Int64,
    "trade_amount_valid": pl.UInt8,
    "trade_amount_is_null": pl.UInt8,
}


def read_retained_fast_ticks_by_symbol(
    client,
    symbol: str,
    symbol_dictionary: dict[str, dict[str, object]],
    *,
    lookback_records: int = 65_536,
    batch_records: int = 4_096,
    maximum_retries: int = 20,
):
    normalized = symbol.strip().upper()
    instrument_id = instrument_id_for(normalized, symbol_dictionary)
    sampled_session = client.session_info()
    frontier_exclusive = (
        sampled_session.tick_contiguous_published_sequence + 1
    )
    requested_first_sequence = max(
        1,
        frontier_exclusive
        - min(lookback_records, sampled_session.tick_ring_capacity),
    )
    next_sequence = requested_first_sequence
    frames_for_symbol: list[pl.DataFrame] = []
    overrun_restarts = 0
    inconsistent_read_retries = 0
    total_retries = 0

    while next_sequence < frontier_exclusive:
        reader = client.open_fast_tick_stream(
            expected_sequence=next_sequence,
            batch_records=batch_records,
        )
        try:
            while reader.next_sequence < frontier_exclusive:
                batch = reader.read(
                    min(
                        batch_records,
                        frontier_exclusive - reader.next_sequence,
                    )
                )
                if not len(batch):
                    break
                values = batch.read_columns(*FAST_TICK_SCHEMA)
                batch_frame = pl.DataFrame(
                    values, schema=FAST_TICK_SCHEMA, strict=True
                ).filter(pl.col("instrument_id") == instrument_id)
                if not batch_frame.is_empty():
                    frames_for_symbol.append(batch_frame)
                next_sequence = batch.next_sequence
            else:
                break
            if reader.next_sequence < frontier_exclusive:
                break
        except TickOverrunError as error:
            overrun_restarts += 1
            total_retries += 1
            if total_retries > maximum_retries:
                raise
            next_sequence = error.observed_sequence
        except InconsistentReadError:
            inconsistent_read_retries += 1
            total_retries += 1
            if total_retries > maximum_retries:
                raise
        finally:
            reader.close()

    fast_ticks = (
        pl.concat(frames_for_symbol, how="vertical", rechunk=True)
        if frames_for_symbol
        else pl.DataFrame(schema=FAST_TICK_SCHEMA)
    )
    fast_ticks = fast_ticks.with_columns(
        pl.lit(normalized, dtype=pl.String).alias("symbol")
    ).select("symbol", *FAST_TICK_SCHEMA)
    scan_metadata = pl.DataFrame(
        {
            "symbol": [normalized],
            "requested_first_sequence": [requested_first_sequence],
            "frontier_exclusive": [frontier_exclusive],
            "overrun_restarts": [overrun_restarts],
            "inconsistent_read_retries": [
                inconsistent_read_retries
            ],
            "retention_truncated_during_scan": [overrun_restarts > 0],
            "returned_symbol_rows": [fast_ticks.height],
        }
    )
    return scan_metadata, fast_ticks


fast_scan, fast_tick_history = read_retained_fast_ticks_by_symbol(
    client, SYMBOL, symbols
)
display(fast_scan)
display(fast_tick_history.tail(5))

latest_fast_trade = (
    fast_tick_history.filter(pl.col("action") == int(TickAction.TRADE))
    .sort("tick_stream_sequence")
    .tail(1)
)
if latest_fast_trade.is_empty():
    print("当前 FAST ring retention 中没有这个 symbol 的成交。")
else:
    print("最近一笔 FAST 成交")
    display(latest_fast_trade)

## 沪市与深市的 Polars 返回值

沪深使用同一套 Wire V2 和固定 Polars schema：对同一个接口，两市的列名、列顺序和 dtype 不会变化，可以直接 `concat`。市场差异由行内的枚举值、validity flags 和数值表达，而不是通过增删列表达。

| 字段/语义 | 沪市 | 深市 |
|---|---|---|
| `market` | `1` (`SHANGHAI`) | `2` (`SHENZHEN`) |
| snapshot `event_kind` | `1` (`SHANGHAI_SNAPSHOT`) | `3` (`SHENZHEN_SNAPSHOT`) |
| tick `event_kind` | `2` (`SHANGHAI_TICK`) | `4` (`SHENZHEN_ORDER`) 或 `5` (`SHENZHEN_TRANSACTION`) |
| `action` | 统一为 `ADD=1 / CANCEL=2 / TRADE=3 / STATUS=4` | 同一套枚举；order 通常是 ADD，transaction 表达 CANCEL/TRADE |
| `side` | 统一枚举；成交可能为 `UNKNOWN=0` | 统一枚举；源数据不能证明方向时同样保持 `UNKNOWN=0` |
| `trade_amount_*` | 成交行可携带源金额 | 深圳 6.36 不提供源金额，`trade_amount_valid=0`；不能把 payload 的 `0` 当作真实成交额，也不能自行用价格乘数量补造 |
| 上海 raw projection | `projection_flags` 及上海 raw type/flag 可有语义 | 同列仍存在，但深圳行不携带上海 raw projection |

FAST/Store raw history 的 `*_valid`、`*_is_null` 是 `UInt8` wire 标志；latest point-read 中对应列是 Polars `Boolean`。读取数值前应先检查 `*_valid`。Derived Event 的列也固定相同，但深圳 `phase`、`aggressor` 和 source trade amount 按源契约保持未知/无效。当前 online-recovery preview 对沪深都拒绝 Store History，这不是市场 schema 差异。

In [ ]:
SH_EXAMPLE_SYMBOL = "SH.600000"
SZ_EXAMPLE_SYMBOL = os.environ.get(
    "L2FLOW_SZ_SYMBOL", "SZ.000001"
).strip().upper()

sh_latest_snapshot_return = latest_snapshot_by_symbol(
    frames, SH_EXAMPLE_SYMBOL, symbols
)
sz_latest_snapshot_return = latest_snapshot_by_symbol(
    frames, SZ_EXAMPLE_SYMBOL, symbols
)
sh_latest_tick_return = latest_tick_by_symbol(
    frames, SH_EXAMPLE_SYMBOL, symbols
)
sz_latest_tick_return = latest_tick_by_symbol(
    frames, SZ_EXAMPLE_SYMBOL, symbols
)

market_schema_comparison = pl.DataFrame(
    {
        "interface": ["latest_snapshot", "latest_tick"],
        "sh_column_count": [
            len(sh_latest_snapshot_return.columns),
            len(sh_latest_tick_return.columns),
        ],
        "sz_column_count": [
            len(sz_latest_snapshot_return.columns),
            len(sz_latest_tick_return.columns),
        ],
        "same_columns": [
            sh_latest_snapshot_return.columns
            == sz_latest_snapshot_return.columns,
            sh_latest_tick_return.columns == sz_latest_tick_return.columns,
        ],
        "same_schema": [
            sh_latest_snapshot_return.schema
            == sz_latest_snapshot_return.schema,
            sh_latest_tick_return.schema == sz_latest_tick_return.schema,
        ],
    }
)
display(market_schema_comparison)

latest_snapshot_market_values = pl.concat(
    [sh_latest_snapshot_return, sz_latest_snapshot_return],
    how="vertical",
).select(
    "symbol",
    "market",
    "event_kind",
    "source_slot",
    "source_stream_id",
    "last_price_p6",
    "last_price_valid",
    "last_price_is_null",
)
display(latest_snapshot_market_values)

latest_tick_market_values = pl.concat(
    [sh_latest_tick_return, sz_latest_tick_return],
    how="vertical",
).select(
    "symbol",
    "market",
    "event_kind",
    "source_slot",
    "source_stream_id",
    "action",
    "side",
    "price_p6",
    "price_valid",
    "quantity_raw",
    "quantity_valid",
    "projection_flags",
)
display(latest_tick_market_values)

In [ ]:
sh_fast_scan, sh_fast_tick_history = (
    read_retained_fast_ticks_by_symbol(
        client, SH_EXAMPLE_SYMBOL, symbols
    )
)
sz_fast_scan, sz_fast_tick_history = (
    read_retained_fast_ticks_by_symbol(
        client, SZ_EXAMPLE_SYMBOL, symbols
    )
)

assert sh_fast_tick_history.columns == sz_fast_tick_history.columns
assert sh_fast_tick_history.schema == sz_fast_tick_history.schema
display(pl.concat([sh_fast_scan, sz_fast_scan], how="vertical"))

fast_market_event_summary = (
    pl.concat(
        [sh_fast_tick_history, sz_fast_tick_history],
        how="vertical",
    )
    .group_by(
        "symbol", "market", "event_kind", "action", "side"
    )
    .len()
    .sort("symbol", "event_kind", "action", "side")
)
display(fast_market_event_summary)

print("深圳 FAST history tail")
display(
    sz_fast_tick_history.select(
        "symbol",
        "market",
        "event_kind",
        "action",
        "side",
        "tick_stream_sequence",
        "price_p6",
        "price_valid",
        "quantity_raw",
        "quantity_valid",
        "trade_amount_p6",
        "trade_amount_valid",
    ).tail(5)
)

market_latest_fast_trades = pl.concat(
    [
        sh_fast_tick_history.filter(
            pl.col("action") == int(TickAction.TRADE)
        ).tail(1),
        sz_fast_tick_history.filter(
            pl.col("action") == int(TickAction.TRADE)
        ).tail(1),
    ],
    how="vertical",
).select(
    "symbol",
    "market",
    "event_kind",
    "tick_stream_sequence",
    "price_p6",
    "price_valid",
    "quantity_raw",
    "quantity_valid",
    "trade_amount_p6",
    "trade_amount_valid",
)
print("沪深最近一笔 FAST 成交")
display(market_latest_fast_trades)

## Instrument Store history：历史 snapshot + tick

`read_store_history_by_symbol()` 接收 symbol，在内部解析 ID 后由 `open_instrument_history()` 固定一个不可变 Store generation。每个 `HistoryPage` 分别暴露 lazy snapshot columns 与 tick columns；下面只选择少量列并转换成两个 Polars DataFrame。`cursor.pages()` 会继续读取独立的 terminal EOF，只有 EOF 与 generation 计数核对成功，调用才算完整。

这里的 tick 表是 Store 原始 tick/order/transaction 投影；derived event 是下一节的另一个产品。

In [ ]:
SNAPSHOT_SCHEMA = {
    "ingress_sequence": pl.UInt64,
    "source_sequence": pl.UInt64,
    "event_time_unix_ns": pl.Int64,
    "recv_realtime_ns": pl.Int64,
    "event_kind": pl.UInt8,
    "market": pl.UInt8,
    "last_price_p6": pl.Int64,
    "last_price_valid": pl.UInt8,
    "last_price_is_null": pl.UInt8,
    "trade_volume_raw": pl.Int64,
    "trade_volume_valid": pl.UInt8,
    "trade_volume_is_null": pl.UInt8,
    "bid_price_1_p6": pl.Int64,
    "bid_quantity_1_raw": pl.Int64,
    "ask_price_1_p6": pl.Int64,
    "ask_quantity_1_raw": pl.Int64,
}

TICK_SCHEMA = {
    "ingress_sequence": pl.UInt64,
    "source_sequence": pl.UInt64,
    "tick_stream_sequence": pl.UInt64,
    "event_time_unix_ns": pl.Int64,
    "recv_realtime_ns": pl.Int64,
    "event_kind": pl.UInt8,
    "market": pl.UInt8,
    "action": pl.UInt8,
    "side": pl.UInt8,
    "price_p6": pl.Int64,
    "price_valid": pl.UInt8,
    "price_is_null": pl.UInt8,
    "quantity_raw": pl.Int64,
    "quantity_valid": pl.UInt8,
    "quantity_is_null": pl.UInt8,
    "trade_amount_p6": pl.Int64,
    "trade_amount_valid": pl.UInt8,
    "trade_amount_is_null": pl.UInt8,
}


def history_columns_frame(columns, schema: dict[str, pl.DataType]) -> pl.DataFrame:
    return pl.DataFrame(
        {name: columns[name] for name in schema},
        schema=schema,
        strict=True,
    )


def concat_history_pages(
    pages: list[pl.DataFrame], schema: dict[str, pl.DataType]
) -> pl.DataFrame:
    if not pages:
        return pl.DataFrame(schema=schema)
    return pl.concat(pages, how="vertical", rechunk=True)


def read_store_history_by_symbol(
    client,
    symbol: str,
    symbol_dictionary: dict[str, dict[str, object]],
):
    normalized = symbol.strip().upper()
    instrument_id = instrument_id_for(normalized, symbol_dictionary)
    snapshot_pages: list[pl.DataFrame] = []
    tick_pages: list[pl.DataFrame] = []

    with client.open_instrument_history(
        instrument_id, requested_page_records=4096
    ) as cursor:
        generation = cursor.generation
        for page in cursor.pages():
            if page.snapshot_count:
                snapshot_pages.append(
                    history_columns_frame(page.snapshot_columns, SNAPSHOT_SCHEMA)
                )
            if page.tick_count:
                tick_pages.append(
                    history_columns_frame(page.tick_columns, TICK_SCHEMA)
                )
        assert cursor.done  # terminal EOF 已读取并通过计数校验

    generation_frame = pl.DataFrame(
        {
            "symbol": [normalized],
            "generation": [generation.generation],
            "trade_date": [generation.trade_date],
            "instrument_id": [generation.instrument_id],
            "record_count": [generation.instrument_record_count],
            "snapshot_count": [generation.snapshot_record_count],
            "tick_count": [generation.tick_record_count],
            "coverage_from_open": [generation.coverage_from_open],
            "record_coverage_complete": [
                generation.record_coverage_complete
            ],
        }
    )
    snapshot_history = concat_history_pages(
        snapshot_pages, SNAPSHOT_SCHEMA
    ).with_columns(
        pl.lit(normalized, dtype=pl.String).alias("symbol")
    ).select("symbol", *SNAPSHOT_SCHEMA)
    tick_history = concat_history_pages(
        tick_pages, TICK_SCHEMA
    ).with_columns(
        pl.lit(normalized, dtype=pl.String).alias("symbol")
    ).select("symbol", *TICK_SCHEMA)
    return generation_frame, snapshot_history, tick_history

In [ ]:
store_generation = None
snapshot_history = pl.DataFrame(schema=SNAPSHOT_SCHEMA)
tick_history = pl.DataFrame(schema=TICK_SCHEMA)

try:
    store_generation, snapshot_history, tick_history = (
        read_store_history_by_symbol(client, SYMBOL, symbols)
    )
except UnavailableError as error:
    print(
        "Store History 当前不可读：",
        error,
        "\n若这是 online recovery preview，此结果符合契约；"
        "等待 promotion 后改连 RECOVERED_CONTROL_SOCKET。",
    )
else:
    display(store_generation)
    print("snapshot history tail")
    display(snapshot_history.tail(5))
    print("tick history tail")
    display(tick_history.tail(5))

## 最近一笔真正的成交

历史 tick 按 Store ingress 顺序返回。筛选 `TickAction.TRADE` 后取最后一行，得到这个固定 generation 截止点内最近的成交；它与上一节的 latest tick-like 热读语义不同。

In [ ]:
latest_trade = (
    tick_history.filter(pl.col("action") == int(TickAction.TRADE))
    .sort("ingress_sequence")
    .tail(1)
)
if latest_trade.is_empty():
    print("当前可读 generation 中没有成交，或当前 History 尚不可用。")
else:
    display(latest_trade)

## Instrument derived event history

`open_derived_event_history_by_symbol()` 接收 symbol，并返回从原始 tick/order/transaction 推导出的 `TRADE`、`CANCEL`、`STATUS` 和带 revision 的 order 状态事件。`wait_ready()` 等待初始完整扫描读到显式 EOF，随后 `latest_snapshot().dataframe` 给出一个固定 Polars cut。

普通 from-open 与 recovered from-open 使用 `coverage_requirement="from_open"`；standalone partial 必须明确使用 `"allow_process_start_partial"`。该参数不会把 partial 数据升级成完整日数据。

In [ ]:
def open_derived_event_history_by_symbol(
    frames, symbol, symbol_dictionary, **kwargs
):
    return frames.open_instrument_derived_event_history(
        instrument_id_for(symbol, symbol_dictionary), **kwargs
    )


derived_history = None
event_history = None
coverage_requirement = (
    "from_open"
    if coverage.coverage_from_open
    else "allow_process_start_partial"
)

try:
    derived_history = open_derived_event_history_by_symbol(
        frames,
        SYMBOL,
        symbols,
        coverage_requirement=coverage_requirement,
        refresh_interval=None,
        live_tail=False,
    )
    derived_history.wait_ready(timeout=10.0)
    event_cut = derived_history.latest_snapshot()
    event_history = event_cut.dataframe.with_columns(
        pl.lit(SYMBOL, dtype=pl.String).alias("symbol")
    ).select("symbol", *event_cut.dataframe.columns)
except (UnavailableError, PolarsHistoryCoverageError) as error:
    if derived_history is not None:
        derived_history.close()
        derived_history = None
    print(
        "Derived Event History 当前不可读：",
        error,
        "\nonline recovery preview 上这是预期结果。",
    )
else:
    print("history coverage:", event_cut.history_coverage)
    display(
        event_history.select(
            "symbol",
            "derived_event_sequence",
            "event_uid",
            "event_kind",
            "operation",
            "tick_stream_sequence",
            "event_time_unix_ns",
            "order_id",
            "side",
            "price_p6",
            "quantity",
        ).tail(10)
    )

## 三种 coverage 下的直接调用示例

主流程通过 `find_and_connect_l2flow_backend()` 自动连接当前可用 endpoint。下面三段是需要精确指定某一种 coverage 时的直接接口示例；每个 endpoint 都先构建自己的 `symbols` 字典，这样不会把一个 session-scoped ID 误用于另一个 session。读取输入始终是 `SYMBOL`，差异主要是 socket、coverage 断言和 derived-event 的 `coverage_requirement`。

### 1. Standalone partial

```python
partial = connect(PARTIAL_SOCKET, **CONNECT_OPTIONS)
partial_coverage = partial.history_coverage()
assert partial_coverage.process_start_partial
partial_symbols, _ = build_symbol_dictionary(partial)
generation, snapshots, ticks = read_store_history_by_symbol(
    partial, SYMBOL, partial_symbols
)
partial_events = open_derived_event_history_by_symbol(
    as_polars(partial), SYMBOL, partial_symbols,
    coverage_requirement="allow_process_start_partial",
)
```

Online CSV recovery 的 preview 不是这个模式：它只开放 latest，History coverage 为 `UNAVAILABLE`。

### 2. 普通 from-open capture

```python
full = connect(FROM_OPEN_SOCKET, **CONNECT_OPTIONS)
full_session = full.session_info()
full_coverage = full.history_coverage()
assert full_coverage.coverage_from_open
assert not full_session.startup_prefix_recovered
full_symbols, _ = build_symbol_dictionary(full)
generation, snapshots, ticks = read_store_history_by_symbol(
    full, SYMBOL, full_symbols
)
full_events = open_derived_event_history_by_symbol(
    as_polars(full), SYMBOL, full_symbols, coverage_requirement="from_open"
)
```

### 3. From-open recovery

Promotion 完成前 recovered socket 保持隐藏；`recovered_control_socket_for(CONTROL_SOCKET)` 会从同一进程的 `--ipc-socket` 参数推导它。完成后重新连接并验证 recovery 标志，读取接口与普通 from-open 相同。

```python
assert RECOVERED_CONTROL_SOCKET is not None
recovered = connect(RECOVERED_CONTROL_SOCKET, **CONNECT_OPTIONS)
recovered_session = recovered.session_info()
recovered_coverage = recovered.history_coverage()
assert recovered_coverage.coverage_from_open
assert recovered_session.startup_prefix_recovered
recovered_symbols, _ = build_symbol_dictionary(recovered)
generation, snapshots, ticks = read_store_history_by_symbol(
    recovered, SYMBOL, recovered_symbols
)
recovered_events = open_derived_event_history_by_symbol(
    as_polars(recovered), SYMBOL, recovered_symbols,
    coverage_requirement="from_open"
)
```

## 探测当前 recovered socket

当前实例仍在 partial recovery 时，下面通常打印“尚未暴露”。promotion 完成后，重新从连接单元开始运行；自动发现会跳过已退役的 preview 并连接 recovered endpoint，此处应看到 `ACTIVE / FROM_OPEN / startup_prefix_recovered=True`。

In [ ]:
recovered_client = None
recovered_session = None
recovered_coverage = None

if RECOVERED_CONTROL_SOCKET is None:
    print("当前 backend 没有独立的 recovered socket。")
elif RECOVERED_CONTROL_SOCKET == CONTROL_SOCKET:
    recovered_session = session
    recovered_coverage = coverage
else:
    try:
        recovered_client = connect(
            str(RECOVERED_CONTROL_SOCKET),
            timeout=1.0,
            **CONNECT_OPTIONS,
        )
        recovered_session = retry_inconsistent_read(
            recovered_client.session_info
        )
        recovered_coverage = retry_inconsistent_read(
            recovered_client.history_coverage
        )
    except Exception as error:
        if recovered_client is not None:
            recovered_client.close()
            recovered_client = None
        print(
            "recovered socket 尚未暴露：",
            f"{type(error).__name__}: {error}",
        )

if recovered_session is not None and recovered_coverage is not None:
    display(
        pl.DataFrame(
            {
                "server_state": [recovered_session.server_state.name],
                "coverage_kind": [recovered_coverage.coverage_kind.name],
                "coverage_from_open": [
                    recovered_coverage.coverage_from_open
                ],
                "startup_prefix_recovered": [
                    recovered_session.startup_prefix_recovered
                ],
            }
        )
    )
    assert recovered_coverage.coverage_from_open
    assert recovered_session.startup_prefix_recovered

## 关闭资源

History handle 带后台线程/native reader，需要显式关闭；顶层 client 也由 notebook 自己关闭。

In [ ]:
if derived_history is not None:
    derived_history.close()
if recovered_client is not None:
    recovered_client.close()
client.close()
print("closed")